# Embeddings — Advanced
> Section 01 — Topic 02 — advanced

**Prerequisites:** 02-embeddings/foundations.ipynb

**What you'll learn:**
- The evolution from static to contextual embeddings
- Modern sentence embedding models and how to choose them
- Similarity metrics and when each is appropriate

**What you'll build:**
- A comprehensive embedding comparison benchmark

## Setup

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt
from transformers import AutoTokenizer, AutoModel

torch.manual_seed(42)
print(f"PyTorch: {torch.__version__}")

## 1. From Static to Contextual Embeddings

Static embeddings like Word2Vec and GloVe assign a **single fixed vector** to each word, regardless of context. The word "bank" gets the same embedding whether it appears in "river bank" or "bank account." This is a fundamental limitation — natural language is deeply contextual, and word meaning shifts constantly based on surrounding words.

**Contextual embeddings** solve this by computing word representations as a function of the entire input sequence. ELMo (2018) was the first major contextual embedding model, using bidirectional LSTMs. BERT (2018) then showed that Transformer-based contextual embeddings were dramatically more powerful. In a Transformer, each layer's output is a contextual embedding — the representation of a token is influenced by every other token in the sequence through self-attention.

The shift from static to contextual embeddings is one of the most important transitions in NLP. Every modern model — GPT, BERT, Llama, Claude — produces contextual embeddings internally. The question now is how to extract and use them effectively.

In [ ]:
# Demonstrate contextual embeddings: same word, different contexts
model_name = "bert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModel.from_pretrained(model_name)
model.eval()

sentences = [
    "I deposited money at the bank",
    "We sat on the river bank",
    "The bank approved my loan application",
    "Fish were jumping near the bank of the stream",
]

# Get contextual embeddings for "bank" in each sentence
bank_embeddings = []
for sent in sentences:
    inputs = tokenizer(sent, return_tensors="pt", padding=True)
    with torch.no_grad():
        outputs = model(**inputs)
    
    # Find the position of "bank"
    tokens = tokenizer.tokenize(sent)
    bank_idx = tokens.index("bank") + 1  # +1 for [CLS]
    
    # Extract embedding from last hidden state
    bank_emb = outputs.last_hidden_state[0, bank_idx]
    bank_embeddings.append(bank_emb)

# Compute pairwise cosine similarity
print("Cosine similarity between 'bank' in different contexts:")
print(f"{'':>50} ", end="")
for i in range(len(sentences)):
    print(f"  S{i+1}", end="")
print()

for i, s1 in enumerate(sentences):
    label = s1[:48].ljust(50)
    print(f"{label} ", end="")
    for j in range(len(sentences)):
        sim = F.cosine_similarity(bank_embeddings[i].unsqueeze(0), bank_embeddings[j].unsqueeze(0)).item()
        print(f" {sim:.2f}", end="")
    print()

print("\nNotice: financial 'bank' (S1,S3) cluster together, river 'bank' (S2,S4) cluster together.")

## 2. Sentence Embeddings

While contextual models give us excellent token-level embeddings, many applications need a single vector for an entire sentence or passage — semantic search, clustering, deduplication, and retrieval all require sentence-level representations.

The naive approach is to take the mean of all token embeddings (mean pooling) or use the [CLS] token embedding. But raw BERT embeddings produce notoriously poor sentence embeddings — Reimers & Gurevych (2019) showed they can be worse than GloVe averages on semantic similarity tasks.

**Sentence-Transformers** fix this by fine-tuning models with a **bi-encoder** architecture: two copies of the same model encode two sentences independently, and a contrastive loss pushes semantically similar sentences together in embedding space. This produces embeddings that are directly comparable with cosine similarity.

In [ ]:
# Three pooling strategies for sentence embeddings

def cls_pooling(model_output):
    """Use [CLS] token embedding as sentence representation."""
    return model_output.last_hidden_state[:, 0]

def mean_pooling(model_output, attention_mask):
    """Average all token embeddings, respecting attention mask."""
    token_embeddings = model_output.last_hidden_state
    input_mask_expanded = attention_mask.unsqueeze(-1).expand(token_embeddings.size()).float()
    return torch.sum(token_embeddings * input_mask_expanded, 1) / torch.clamp(input_mask_expanded.sum(1), min=1e-9)

def max_pooling(model_output, attention_mask):
    """Take element-wise max across all token embeddings."""
    token_embeddings = model_output.last_hidden_state
    input_mask_expanded = attention_mask.unsqueeze(-1).expand(token_embeddings.size()).float()
    token_embeddings[input_mask_expanded == 0] = -1e9
    return torch.max(token_embeddings, 1)[0]

# Compare pooling strategies on semantic similarity
pairs = [
    ("A dog is playing in the park", "A puppy runs around the garden"),     # Similar
    ("A dog is playing in the park", "The stock market crashed today"),      # Different
    ("Python is a programming language", "Python is a type of snake"),       # Ambiguous
    ("The food was excellent", "The meal was outstanding"),                  # Similar
]

print(f"{'Pooling':<12} {'Pair':<55} {'Similarity':>10}")
print("-" * 80)

for pool_name, pool_fn in [("CLS", lambda o, m: cls_pooling(o)), 
                            ("Mean", mean_pooling), 
                            ("Max", max_pooling)]:
    for s1, s2 in pairs:
        inputs1 = tokenizer(s1, return_tensors="pt", padding=True, truncation=True)
        inputs2 = tokenizer(s2, return_tensors="pt", padding=True, truncation=True)
        
        with torch.no_grad():
            out1 = model(**inputs1)
            out2 = model(**inputs2)
        
        emb1 = pool_fn(out1, inputs1['attention_mask'])
        emb2 = pool_fn(out2, inputs2['attention_mask'])
        
        sim = F.cosine_similarity(emb1, emb2).item()
        label = f"{s1[:25]}... / {s2[:25]}..."
        print(f"{pool_name:<12} {label:<55} {sim:>10.3f}")
    print()

In [ ]:
# Using sentence-transformers for proper sentence embeddings
try:
    from sentence_transformers import SentenceTransformer
    
    st_model = SentenceTransformer('all-MiniLM-L6-v2')
    
    sentences = [
        "A dog is playing in the park",
        "A puppy runs around the garden",
        "The stock market crashed today",
        "Financial markets experienced a downturn",
        "Python is great for data science",
    ]
    
    embeddings = st_model.encode(sentences, convert_to_tensor=True)
    
    # Similarity matrix
    sim_matrix = F.cosine_similarity(
        embeddings.unsqueeze(0), embeddings.unsqueeze(1), dim=-1
    ).numpy()
    
    fig, ax = plt.subplots(figsize=(8, 6))
    im = ax.imshow(sim_matrix, cmap='RdYlGn', vmin=-1, vmax=1)
    ax.set_xticks(range(len(sentences)))
    ax.set_yticks(range(len(sentences)))
    short_labels = [s[:30] + "..." if len(s) > 30 else s for s in sentences]
    ax.set_xticklabels(short_labels, rotation=45, ha='right', fontsize=8)
    ax.set_yticklabels(short_labels, fontsize=8)
    
    for i in range(len(sentences)):
        for j in range(len(sentences)):
            ax.text(j, i, f"{sim_matrix[i,j]:.2f}", ha='center', va='center', fontsize=9)
    
    plt.colorbar(im, ax=ax)
    ax.set_title('Sentence Similarity (all-MiniLM-L6-v2)')
    plt.tight_layout()
    plt.show()
    
except ImportError:
    print("Install sentence-transformers: pip install sentence-transformers")
    print("Then re-run this cell.")

## 3. Modern Embedding Models

The embedding model landscape has exploded since 2023. The MTEB (Massive Text Embedding Benchmark) leaderboard tracks hundreds of models across dozens of tasks. Here are the key families:

- **BGE (BAAI General Embedding):** Open-source models from BAAI, consistently strong across tasks. bge-large-en-v1.5 was the top open model for months.
- **E5:** Microsoft's embedding models, trained with a novel weakly-supervised approach using (query, passage) pairs scraped from the web.
- **GTE (General Text Embeddings):** Alibaba's models, competitive with BGE.
- **Jina Embeddings:** Specializes in long-context embeddings (up to 8K tokens).
- **Nomic Embed:** Open-source, 8K context, strong performance with small model size.
- **OpenAI text-embedding-3:** Commercial API, offers small and large variants with configurable dimensions.
- **Cohere Embed v3:** Commercial, supports multiple input types (search_query vs search_document).

The MTEB benchmark evaluates models across 7 task categories: classification, clustering, pair classification, reranking, retrieval, STS (semantic textual similarity), and summarization. No single model dominates all categories — task-specific selection matters.

In [ ]:
# Compare multiple open-source embedding models
model_configs = {
    "all-MiniLM-L6-v2": {"dim": 384, "max_seq": 256, "params": "22M"},
    "all-mpnet-base-v2": {"dim": 768, "max_seq": 384, "params": "109M"},
    "bge-small-en-v1.5": {"dim": 384, "max_seq": 512, "params": "33M"},
    "bge-base-en-v1.5": {"dim": 768, "max_seq": 512, "params": "109M"},
    "nomic-embed-text-v1.5": {"dim": 768, "max_seq": 8192, "params": "137M"},
}

# Summary table
print(f"{'Model':<30} {'Dim':>5} {'Max Seq':>8} {'Params':>8}")
print("-" * 55)
for name, cfg in model_configs.items():
    print(f"{name:<30} {cfg['dim']:>5} {cfg['max_seq']:>8} {cfg['params']:>8}")

print("\nKey tradeoffs:")
print("  - Higher dim = more expressive but slower retrieval and more storage")
print("  - Longer max_seq = handles longer documents but slower encoding")
print("  - More params = generally better quality but slower inference")

In [ ]:
# Benchmark embedding models on a retrieval task
try:
    from sentence_transformers import SentenceTransformer
    import time
    
    # Test queries and documents
    queries = [
        "How does gradient descent work?",
        "What is the capital of France?",
        "How to train a neural network?",
    ]
    
    documents = [
        "Gradient descent is an optimization algorithm that iteratively adjusts parameters by moving in the direction of steepest decrease of the loss function.",
        "Paris is the capital and most populous city of France, situated on the Seine river.",
        "Training a neural network involves forward propagation, loss computation, backpropagation, and weight updates using an optimizer like Adam or SGD.",
        "The Eiffel Tower is a wrought-iron lattice tower on the Champ de Mars in Paris.",
        "Backpropagation computes gradients of the loss with respect to each weight by applying the chain rule.",
        "Machine learning models learn patterns from data through iterative optimization.",
    ]
    
    models_to_test = ["all-MiniLM-L6-v2", "all-mpnet-base-v2"]
    
    for model_name in models_to_test:
        st = SentenceTransformer(model_name)
        
        start = time.perf_counter()
        q_emb = st.encode(queries, convert_to_tensor=True)
        d_emb = st.encode(documents, convert_to_tensor=True)
        encode_time = time.perf_counter() - start
        
        # Compute similarity
        sims = F.cosine_similarity(q_emb.unsqueeze(1), d_emb.unsqueeze(0), dim=-1)
        
        print(f"\n=== {model_name} (encode: {encode_time:.2f}s) ===")
        for i, query in enumerate(queries):
            ranked = sims[i].argsort(descending=True)
            print(f"  Q: {query}")
            for rank, idx in enumerate(ranked[:2]):
                print(f"    #{rank+1} ({sims[i][idx]:.3f}): {documents[idx][:70]}...")

except ImportError:
    print("Install sentence-transformers to run this benchmark")

## 4. Choosing Embedding Models

Model selection depends heavily on your specific use case. The key factors are:

**Task type:** Retrieval models are trained differently from classification models. A model that excels at semantic search may underperform at clustering. The MTEB benchmark breaks performance down by task category — check the relevant category, not just the overall score.

**Dimension tradeoff:** Higher-dimensional embeddings capture more information but require more storage and slower similarity computation. For a vector database with millions of documents, going from 768 to 384 dimensions halves your storage cost and roughly doubles search speed. OpenAI's text-embedding-3 models support dimension reduction via the `dimensions` parameter.

**Context length:** Most embedding models truncate at 512 tokens. For long documents, you need either a long-context model (Jina, Nomic at 8K) or a chunking strategy.

**Inference cost:** Open-source models on your hardware have zero marginal cost but require GPU memory. API models (OpenAI, Cohere) charge per token but require no infrastructure.

In [ ]:
# Decision framework for embedding model selection
import json

decision_tree = {
    "Semantic Search / RAG": {
        "budget": "bge-base-en-v1.5 or all-MiniLM-L6-v2",
        "quality": "bge-large-en-v1.5 or nomic-embed-text-v1.5",
        "long_docs": "nomic-embed-text-v1.5 (8K context) or jina-embeddings-v2",
        "api": "OpenAI text-embedding-3-large",
    },
    "Classification": {
        "budget": "all-MiniLM-L6-v2",
        "quality": "all-mpnet-base-v2",
    },
    "Clustering": {
        "budget": "all-MiniLM-L6-v2",
        "quality": "bge-base-en-v1.5",
    },
    "Multilingual": {
        "any": "multilingual-e5-large or paraphrase-multilingual-mpnet-base-v2",
    },
}

for task, options in decision_tree.items():
    print(f"\n{task}:")
    for constraint, model in options.items():
        print(f"  [{constraint}] → {model}")

## 5. Embedding Similarity Metrics

Three metrics dominate embedding similarity computation, and the choice matters more than most people realize:

**Cosine similarity** measures the angle between two vectors, ignoring magnitude: $\cos(\theta) = \frac{a \cdot b}{\|a\| \|b\|}$. Range: [-1, 1]. This is the default for most embedding models because it's invariant to vector length — a longer document doesn't automatically appear more similar.

**Dot product** is simply $a \cdot b = \sum a_i b_i$. It's sensitive to magnitude, which can be a feature or a bug. When embeddings are L2-normalized, dot product equals cosine similarity. Some models (like OpenAI's) normalize embeddings, making the choice irrelevant.

**Euclidean distance** measures straight-line distance: $\|a - b\|_2 = \sqrt{\sum(a_i - b_i)^2}$. Smaller = more similar. It's sensitive to magnitude and affected by the curse of dimensionality (in very high dimensions, distances converge). For normalized vectors, Euclidean distance is a monotonic transformation of cosine similarity: $\|a - b\|^2 = 2 - 2\cos(\theta)$.

In [ ]:
# Implement all three metrics and show when they diverge

def cosine_sim(a, b):
    return torch.dot(a, b) / (torch.norm(a) * torch.norm(b))

def dot_product(a, b):
    return torch.dot(a, b)

def euclidean_dist(a, b):
    return torch.norm(a - b)

# Case 1: Normalized vectors — all metrics agree on ranking
print("=== Case 1: Normalized Vectors ===")
a = F.normalize(torch.randn(64), dim=0)
b = F.normalize(torch.randn(64), dim=0)
c = F.normalize(a + 0.1 * torch.randn(64), dim=0)  # Similar to a

print(f"  a vs b: cos={cosine_sim(a,b):.3f}, dot={dot_product(a,b):.3f}, euclid={euclidean_dist(a,b):.3f}")
print(f"  a vs c: cos={cosine_sim(a,c):.3f}, dot={dot_product(a,c):.3f}, euclid={euclidean_dist(a,c):.3f}")
print(f"  Rankings agree: c is closer to a by all metrics")

# Case 2: Unnormalized vectors — dot product diverges
print("\n=== Case 2: Unnormalized Vectors (metrics disagree!) ===")
a = torch.randn(64)
b = a * 0.5 + torch.randn(64) * 0.1  # Similar direction, different magnitude
c = torch.randn(64) * 10              # Random direction, large magnitude

print(f"  a vs b: cos={cosine_sim(a,b):.3f}, dot={dot_product(a,b):.3f}, euclid={euclidean_dist(a,b):.3f}")
print(f"  a vs c: cos={cosine_sim(a,c):.3f}, dot={dot_product(a,c):.3f}, euclid={euclidean_dist(a,c):.3f}")
print(f"  Cosine says b is closer (same direction), but dot product may prefer c (large magnitude)")
print(f"  This is why cosine similarity is preferred when embeddings aren't normalized")

In [ ]:
# Visualize the relationship between metrics for normalized vs unnormalized
n_pairs = 500
dim = 128

# Generate random vector pairs
vecs_a = torch.randn(n_pairs, dim)
vecs_b = torch.randn(n_pairs, dim)

# Compute metrics - unnormalized
cos_sims = F.cosine_similarity(vecs_a, vecs_b).numpy()
dots = (vecs_a * vecs_b).sum(dim=1).numpy()
euclids = torch.norm(vecs_a - vecs_b, dim=1).numpy()

# Normalized versions
vecs_a_norm = F.normalize(vecs_a, dim=1)
vecs_b_norm = F.normalize(vecs_b, dim=1)
cos_norm = F.cosine_similarity(vecs_a_norm, vecs_b_norm).numpy()
dots_norm = (vecs_a_norm * vecs_b_norm).sum(dim=1).numpy()
euclids_norm = torch.norm(vecs_a_norm - vecs_b_norm, dim=1).numpy()

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

axes[0].scatter(cos_sims, dots, alpha=0.3, s=5, label='Unnormalized')
axes[0].scatter(cos_norm, dots_norm, alpha=0.3, s=5, label='Normalized')
axes[0].set_xlabel('Cosine Similarity')
axes[0].set_ylabel('Dot Product')
axes[0].legend(fontsize=8)
axes[0].set_title('Cosine vs Dot Product')

axes[1].scatter(cos_sims, euclids, alpha=0.3, s=5, label='Unnormalized')
axes[1].scatter(cos_norm, euclids_norm, alpha=0.3, s=5, label='Normalized')
axes[1].set_xlabel('Cosine Similarity')
axes[1].set_ylabel('Euclidean Distance')
axes[1].legend(fontsize=8)
axes[1].set_title('Cosine vs Euclidean')

axes[2].scatter(dots, euclids, alpha=0.3, s=5, label='Unnormalized')
axes[2].scatter(dots_norm, euclids_norm, alpha=0.3, s=5, label='Normalized')
axes[2].set_xlabel('Dot Product')
axes[2].set_ylabel('Euclidean Distance')
axes[2].legend(fontsize=8)
axes[2].set_title('Dot Product vs Euclidean')

fig.suptitle('Metric Relationships: Normalized vectors make all metrics equivalent', y=1.02)
plt.tight_layout()
plt.show()

## Summary

**Key takeaways:**
- Contextual embeddings produce different vectors for the same word based on context — a massive improvement over static embeddings
- Sentence-transformers fine-tune models specifically for sentence-level similarity with bi-encoder architecture
- No single embedding model wins everywhere — choose based on task, dimension needs, context length, and cost
- For normalized embeddings (most modern models), cosine similarity = dot product — the metric choice is irrelevant
- For unnormalized embeddings, cosine similarity is safer as it ignores magnitude

**Next:** [02-embeddings/expert.ipynb](expert.ipynb) — Embedding space geometry, probing, Matryoshka embeddings, quantization